# EDA — Dataset maestro híbrido de Taxi IA Chapinero

**Unidad de análisis:** una única tabla de datos, pero **dos orígenes**: registros reales de Bitcarrier (tramo TID, cada 15 minutos) y escenarios simulados para OSM (arista, día de semana, hora). Esta distinción se conserva en cada fila. El EDA permite observar el conjunto completo y comparar estratos de origen para no confundir frecuencias.

**Advertencia metodológica:** los tiempos son derivados; las unidades `DISTANCE = metros` y `VEL_PROMEDIO = km/h` son supuestos pendientes de validación. La serie histórica corresponde a mayo de 2022 y no representa tráfico actual. Ningún emparejamiento Bitcarrier–OSM está validado.

## 1. Lectura reproducible de la fuente única

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ruta_env = os.environ.get("PROYECTO_IA_RAIZ")
if ruta_env:
    raiz = Path(ruta_env).resolve()
else:
    candidatos = [Path.cwd(), *Path.cwd().parents]
    raiz = next(
        (
            p
            for p in candidatos
            if (
                p / "backend/datos/ml/dataset_maestro_hibrido/dataset_maestro_hibrido.csv.gz"
            ).exists()
        ),
        None,
    )
    if raiz is None:
        raise FileNotFoundError(
            "Ejecuta primero construir_dataset_maestro.py desde la raíz del repositorio."
        )
ruta = raiz / "backend/datos/ml/dataset_maestro_hibrido/dataset_maestro_hibrido.csv.gz"
df = pd.read_csv(
    ruta, low_memory=False, dtype={"id_espacial": "string", "tid_bitcarrier": "string"}
)
print("Origen:", ruta)
print("Dimensión:", df.shape)
print("Columnas:", df.columns.tolist())
df.head(4)

## 2. Calidad, tipos y procedencia

In [ ]:
print("Registros por fuente:")
print(df["fuente_dato"].value_counts().to_string())
print("\nGranularidad por fuente:")
print(pd.crosstab(df["fuente_dato"], df["granularidad_min"]))
print("\nNulos por columna:")
print(df.isna().sum().sort_values(ascending=False).to_string())
print("\nIDs de registro duplicados:", int(df["id_registro"].duplicated().sum()))
print("Tiempos no positivos:", int((df["tiempo_min"] <= 0).sum()))
print("Velocidades >100:", int((df["velocidad_kmh"] > 100).sum()))

**Interpretación:** los nulos de `tid_bitcarrier` en OSM, y de `id_arista_osm` en Bitcarrier, son **ausencias estructurales esperadas** porque no tenemos correspondencias verificadas. No se deben imputar con identificadores arbitrarios. La comparación de fuentes no valida por sí sola los valores simulados.

## 3. Estadísticas descriptivas del dataset maestro y por origen

In [ ]:
cols = ["distancia_m", "velocidad_kmh", "tiempo_min"]
display(df[cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(3))
display(df.groupby("fuente_dato")[cols].agg(["count", "median", "mean", "min", "max"]).round(3))

La mediana conjunta depende de la cantidad de escenarios simulados y observaciones históricas; no debe interpretarse como un valor medido de toda Chapinero.

## 4. Procedencia del dataset

In [ ]:
conteos = df["fuente_dato"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    ["Bitcarrier histórico", "OSM simulado"],
    [
        conteos.get("HISTORICO_BITCARRIER_2022", 0),
        conteos.get("SIMULADO_CALIBRADO_ENTRENAMIENTO", 0),
    ],
)
ax.set_ylabel("Número de filas")
ax.set_title("Composición del dataset maestro")
plt.tight_layout()
plt.show()

## 5. Distribución de velocidades e identificación de extremos

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for nombre, s in df.groupby("fuente_dato"):
    ax.hist(s["velocidad_kmh"], bins=np.arange(0, 181, 5), alpha=0.5, label=nombre)
ax.set_xlabel("Velocidad (unidad km/h asumida)")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.show()
print("Velocidades >100 por fuente:")
print(df.assign(mayor_100=df.velocidad_kmh > 100).groupby("fuente_dato")["mayor_100"].sum())

## 6. Patrones horarios y semanales por procedencia

In [ ]:
res = df.groupby(["fuente_dato", "hora"], observed=True).velocidad_kmh.median().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
for nombre, z in res.groupby("fuente_dato"):
    ax.plot(z.hora, z.velocidad_kmh, marker="o", label=nombre)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora declarada")
ax.set_ylabel("Velocidad mediana")
ax.legend()
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9, 4))
por_dia = df.groupby(["fuente_dato", "dia_semana"], observed=True).velocidad_kmh.median().unstack(0)
por_dia.plot(kind="bar", ax=ax)
ax.set_xlabel("Día (0=lunes)")
ax.set_ylabel("Mediana de velocidad")
plt.tight_layout()
plt.show()

## 7. Mapa de calor de escenarios OSM

In [ ]:
sim = df[df.fuente_dato.eq("SIMULADO_CALIBRADO_ENTRENAMIENTO")]
matriz = sim.pivot_table(
    index="dia_semana", columns="hora", values="velocidad_kmh", aggfunc="median"
)
fig, ax = plt.subplots(figsize=(12, 4))
img = ax.imshow(matriz, aspect="auto")
ax.set_yticks(range(7), ["Lun", "Mar", "Mie", "Jue", "Vie", "Sab", "Dom"])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora")
ax.set_title("Simulación OSM — mediana por día y hora")
fig.colorbar(img, ax=ax, label="Velocidad mediana")
plt.tight_layout()
plt.show()

## 8. Distribución de tiempos derivados

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for nombre, s in df.groupby("fuente_dato"):
    ax.hist(
        s["tiempo_min"].clip(upper=s["tiempo_min"].quantile(0.99)),
        bins=45,
        alpha=0.45,
        label=nombre,
    )
ax.set_xlabel("Tiempo derivado en minutos (cola truncada para visualización al p99 de cada fuente)")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Particiones temporales de observaciones históricas

In [ ]:
real = df[df.fuente_dato.eq("HISTORICO_BITCARRIER_2022")]
print(real["particion"].value_counts().to_string())
print("\nTID diferentes:", real["tid_bitcarrier"].nunique())
print("Aristas OSM diferentes en simulación:", sim["indice_arista_osm"].nunique())
print("Combinaciones semana tipo simulada:", sim[["dia_semana", "hora"]].drop_duplicates().shape[0])

## 10. Conclusiones y criterios para los modelos

1. El archivo maestro combina **registros históricos** y **escenarios simulados** con trazabilidad; ambos pueden explorarse en un solo notebook y dashboard.
2. La proporción simulada domina ciertas agregaciones porque cada arista tiene 168 escenarios; comparar ambas fuentes por separado ayuda a interpretar la distribución conjunta.
3. **La regresión de tiempo requiere `y = tiempo_min`**, pero `tiempo_min` deriva de distancia y velocidad, no es una duración medida directamente.
4. **Fuga de información:** jamás incluir `velocidad_kmh` como predictor del tiempo de la misma fila. Si se usa como `y` alternativa, definirlo expresamente.
5. El historial se divide en entrenamiento (1–21), validación (22–26), y prueba (27–31 de mayo). La simulación está calibrada SOLO con el tramo de entrenamiento.
6. Las métricas sobre filas simuladas **no** demuestran generalización real. Para evaluar sobre el tráfico observado usar exclusivamente `fuente_dato == HISTORICO_BITCARRIER_2022` en `particion == prueba`.
7. Los registros Bitcarrier están por `TID`, los escenarios por arista OSM; aún faltan correspondencias espaciales verificadas. No se puede declarar que los valores históricos midieron esas aristas.
8. Es obligatorio confirmar las unidades del proveedor antes de publicar duraciones en minutos como si fueran verificadas.
